# 🔬 Đánh Giá Mô Hình Recommendation

Notebook này đánh giá và so sánh các phương pháp recommendation khác nhau:
- **ALS** (Alternating Least Squares) - Collaborative Filtering
- **Session-based** - Dựa trên hành vi gần đây
- **Vector-based** - Semantic similarity
- **Hybrid** - Kết hợp nhiều phương pháp

## 📊 Metrics được đánh giá:
- Precision@K, Recall@K, NDCG@K
- Mean Reciprocal Rank (MRR)
- Coverage
- Diversity

## 1. Setup và Import

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any, Set, Tuple
from collections import defaultdict
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Packages imported successfully!")

In [ ]:
# Import project modules (optional - requires backend configuration)
try:
    from offline.evaluation.offline_metrics import (
        precision_at_k,
        recall_at_k,
        ndcg_at_k,
        mean_reciprocal_rank,
        coverage,
        diversity,
    )
    print("✅ Project modules imported successfully!")
except ImportError as e:
    print(f"⚠️ Import error: {e}")
    print("Note: Will use local implementations of metrics")

## 2. Load và Khám Phá Dữ Liệu

In [ ]:
# Configuration
DATA_PATH = "../data/raw/user_interactions.csv"
SAMPLE_SIZE = None  # Set to None to use all data, or a number to sample

print(f"📂 Loading data from: {DATA_PATH}")

# Load CSV file
if SAMPLE_SIZE:
    df = pd.read_csv(DATA_PATH, nrows=SAMPLE_SIZE)
else:
    df = pd.read_csv(DATA_PATH)

print(f"✅ Loaded {len(df):,} rows")
print(f"📊 Shape: {df.shape}")
print(f"\n📋 Columns: {df.columns.tolist()}")

In [ ]:
# Display first few rows
print("\n📄 First 10 rows:")
display(df.head(10))

In [ ]:
# Data info
print("\n📊 Data Info:")
print(df.info())
print("\n📈 Basic Statistics:")
display(df.describe())

In [ ]:
# Check for missing values
print("\n❓ Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0])

# Check for duplicate rows
print(f"\n🔄 Duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Identify key columns (common column names)
column_mapping = {
    'user_id': None,
    'product_id': None,
    'timestamp': None,
    'event_type': None,
    'rating': None,
    'score': None,
}

# Try to find matching columns (case-insensitive)
df_columns_lower = {col.lower(): col for col in df.columns}
for key in column_mapping:
    if key in df_columns_lower:
        column_mapping[key] = df_columns_lower[key]
    elif key.replace('_', '') in df_columns_lower:
        column_mapping[key] = df_columns_lower[key.replace('_', '')]

print("🔍 Column Mapping:")
for key, value in column_mapping.items():
    print(f"  {key}: {value}")

# Validate required columns
if not column_mapping['user_id'] or not column_mapping['product_id']:
    print("\n❌ ERROR: Could not find user_id and/or product_id columns!")
    print(f"Available columns: {df.columns.tolist()}")
else:
    print("\n✅ Required columns found!")

In [ ]:
# Basic statistics about users and products
user_col = column_mapping['user_id']
product_col = column_mapping['product_id']

if user_col and product_col:
    print(f"👥 Unique users: {df[user_col].nunique():,}")
    print(f"📦 Unique products: {df[product_col].nunique():,}")
    print(f"📊 Total interactions: {len(df):,}")
    print(f"📈 Avg interactions per user: {len(df) / df[user_col].nunique():.2f}")
    print(f"📈 Avg interactions per product: {len(df) / df[product_col].nunique():.2f}")
    
    # Distribution of interactions per user
    user_counts = df.groupby(user_col).size()
    print(f"\n📊 User interaction statistics:")
    print(f"  Min: {user_counts.min()}")
    print(f"  Max: {user_counts.max()}")
    print(f"  Mean: {user_counts.mean():.2f}")
    print(f"  Median: {user_counts.median():.2f}")
    
    # Distribution of interactions per product
    product_counts = df.groupby(product_col).size()
    print(f"\n📊 Product interaction statistics:")
    print(f"  Min: {product_counts.min()}")
    print(f"  Max: {product_counts.max()}")
    print(f"  Mean: {product_counts.mean():.2f}")
    print(f"  Median: {product_counts.median():.2f}")

In [ ]:
# Visualize data distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# User interaction counts (log scale)
if user_col:
    user_counts = df.groupby(user_col).size()
    axes[0, 0].hist(user_counts, bins=50, edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('Number of Interactions per User')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Distribution of User Interactions')
    axes[0, 0].set_yscale('log')

# Product interaction counts (log scale)
if product_col:
    product_counts = df.groupby(product_col).size()
    axes[0, 1].hist(product_counts, bins=50, edgecolor='black', alpha=0.7, color='orange')
    axes[0, 1].set_xlabel('Number of Interactions per Product')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Distribution of Product Interactions')
    axes[0, 1].set_yscale('log')

# Top users
if user_col:
    top_users = user_counts.nlargest(20)
    axes[1, 0].barh(range(len(top_users)), top_users.values)
    axes[1, 0].set_yticks(range(len(top_users)))
    axes[1, 0].set_yticklabels([f"User {i}" for i in top_users.index[:20]], fontsize=8)
    axes[1, 0].set_xlabel('Number of Interactions')
    axes[1, 0].set_title('Top 20 Most Active Users')
    axes[1, 0].invert_yaxis()

# Top products
if product_col:
    top_products = product_counts.nlargest(20)
    axes[1, 1].barh(range(len(top_products)), top_products.values, color='green')
    axes[1, 1].set_yticks(range(len(top_products)))
    axes[1, 1].set_yticklabels([f"Product {i}" for i in top_products.index[:20]], fontsize=8)
    axes[1, 1].set_xlabel('Number of Interactions')
    axes[1, 1].set_title('Top 20 Most Popular Products')
    axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

## 3. Chuẩn Bị Dữ Liệu cho Đánh Giá

In [ ]:
def prepare_interaction_data(df: pd.DataFrame, user_col: str, product_col: str, 
                            timestamp_col: str = None) -> pd.DataFrame:
    """
    Chuẩn bị dữ liệu interaction cho đánh giá.
    
    Returns DataFrame với columns: user_id, product_id, timestamp
    """
    data = pd.DataFrame()
    data['user_id'] = df[user_col].astype(str)
    data['product_id'] = df[product_col].astype(str)
    
    if timestamp_col and timestamp_col in df.columns:
        data['timestamp'] = pd.to_datetime(df[timestamp_col], errors='coerce')
        data = data.sort_values('timestamp').reset_index(drop=True)
    else:
        data['timestamp'] = pd.date_range(start='2020-01-01', periods=len(data), freq='H')
    
    data = data.drop_duplicates(subset=['user_id', 'product_id'], keep='first')
    
    return data

# Prepare data
timestamp_col = column_mapping.get('timestamp')
interactions_df = prepare_interaction_data(df, user_col, product_col, timestamp_col)

print(f"✅ Prepared {len(interactions_df):,} unique user-product interactions")
print(f"👥 Unique users: {interactions_df['user_id'].nunique():,}")
print(f"📦 Unique products: {interactions_df['product_id'].nunique():,}")

In [ ]:
def split_train_test_user_based(df: pd.DataFrame, test_ratio: float = 0.2,
                                min_interactions: int = 3) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Set[str]]]:
    """
    Split data thành train và test: với mỗi user, giữ lại một số interactions cho test.
    """
    user_counts = df.groupby('user_id').size()
    valid_users = user_counts[user_counts >= min_interactions].index
    df_filtered = df[df['user_id'].isin(valid_users)].copy()
    
    train_list = []
    test_list = []
    test_ground_truth = {}
    
    for user_id in df_filtered['user_id'].unique():
        user_df = df_filtered[df_filtered['user_id'] == user_id].sort_values('timestamp').reset_index(drop=True)
        
        split_idx = int(len(user_df) * (1 - test_ratio))
        
        if split_idx > 0 and split_idx < len(user_df):
            train_list.append(user_df.iloc[:split_idx])
            test_list.append(user_df.iloc[split_idx:])
            
            test_products = set(user_df.iloc[split_idx:]['product_id'].unique())
            if test_products:
                test_ground_truth[str(user_id)] = test_products
    
    train_df = pd.concat(train_list, ignore_index=True)
    test_df = pd.concat(test_list, ignore_index=True)
    
    print(f"📊 Train set: {len(train_df):,} interactions ({len(train_df['user_id'].unique()):,} users)")
    print(f"📊 Test set: {len(test_df):,} interactions ({len(test_df['user_id'].unique()):,} users)")
    print(f"📊 Test ground truth: {len(test_ground_truth):,} users with test interactions")
    
    return train_df, test_df, test_ground_truth

# Split data
train_df, test_df, test_ground_truth = split_train_test_user_based(
    interactions_df, test_ratio=0.2, min_interactions=3
)

## 4. Baseline Models (Simple Recommendation Strategies)

In [ ]:
class BaselineRecommender:
    """Simple baseline recommendation methods for comparison."""
    
    def __init__(self, train_df: pd.DataFrame):
        self.train_df = train_df
        self.popular_items = train_df.groupby('product_id').size().sort_values(ascending=False)
        self.user_items = train_df.groupby('user_id')['product_id'].apply(set).to_dict()
        
        # Item co-occurrence (for collaborative filtering baseline)
        self.item_cooccurrence = defaultdict(lambda: defaultdict(int))
        for user_id, items in self.user_items.items():
            items_list = list(items)
            for i in range(len(items_list)):
                for j in range(i+1, len(items_list)):
                    self.item_cooccurrence[items_list[i]][items_list[j]] += 1
                    self.item_cooccurrence[items_list[j]][items_list[i]] += 1
    
    def recommend_popular(self, user_id: str, k: int = 10, exclude: Set[str] = None) -> List[str]:
        """Recommend most popular items."""
        exclude = exclude or set()
        recommended = [pid for pid in self.popular_items.index if pid not in exclude]
        return recommended[:k]
    
    def recommend_user_based_cf(self, user_id: str, k: int = 10, exclude: Set[str] = None) -> List[str]:
        """Simple collaborative filtering based on item co-occurrence."""
        exclude = exclude or set()
        
        if user_id not in self.user_items:
            return self.recommend_popular(user_id, k, exclude)
        
        user_items = self.user_items[user_id]
        scores = defaultdict(float)
        
        for item in user_items:
            for related_item, count in self.item_cooccurrence[item].items():
                if related_item not in user_items and related_item not in exclude:
                    scores[related_item] += count
        
        recommended = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [pid for pid, _ in recommended[:k]]

# Initialize baseline recommender
baseline = BaselineRecommender(train_df)
print("✅ Baseline recommender initialized")

## 5. Evaluation Functions

In [ ]:
def evaluate_recommendations(
    recommendations: List[str],
    ground_truth: Set[str],
    k_values: List[int] = [5, 10, 20]
) -> Dict[str, float]:
    """
    Evaluate recommendations against ground truth.
    
    Returns dictionary of metrics.
    """
    metrics = {}
    
    for k in k_values:
        top_k = recommendations[:k] if recommendations else []
        
        # Precision@K
        if top_k:
            relevant_count = sum(1 for item in top_k if item in ground_truth)
            metrics[f'precision@{k}'] = relevant_count / len(top_k)
        else:
            metrics[f'precision@{k}'] = 0.0
        
        # Recall@K
        if ground_truth:
            relevant_count = sum(1 for item in top_k if item in ground_truth)
            metrics[f'recall@{k}'] = relevant_count / len(ground_truth)
        else:
            metrics[f'recall@{k}'] = 0.0
        
        # NDCG@K
        dcg = 0.0
        for i, item in enumerate(top_k):
            if item in ground_truth:
                dcg += 1.0 / np.log2(i + 2)
        
        idcg = 0.0
        num_relevant = min(len(ground_truth), k)
        for i in range(num_relevant):
            idcg += 1.0 / np.log2(i + 2)
        
        metrics[f'ndcg@{k}'] = dcg / idcg if idcg > 0 else 0.0
    
    # MRR
    for i, item in enumerate(recommendations):
        if item in ground_truth:
            metrics['mrr'] = 1.0 / (i + 1)
            break
    else:
        metrics['mrr'] = 0.0
    
    return metrics


def evaluate_recommender(
    recommender_func: callable,
    test_ground_truth: Dict[str, Set[str]],
    train_user_items: Dict[str, Set[str]] = None,
    k_values: List[int] = [5, 10, 20],
    max_users: int = None
) -> Dict[str, float]:
    """
    Evaluate a recommendation function on test set.
    """
    all_metrics = defaultdict(list)
    
    user_ids = list(test_ground_truth.keys())
    if max_users:
        user_ids = user_ids[:max_users]
    
    k_max = max(k_values) if k_values else 20
    
    for user_id in user_ids:
        ground_truth = test_ground_truth[user_id]
        
        try:
            exclude = train_user_items.get(user_id, set()) if train_user_items else set()
            recommendations = recommender_func(user_id, k_max, exclude)
            
            if not recommendations:
                continue
            
            metrics = evaluate_recommendations(recommendations, ground_truth, k_values)
            
            for metric_name, value in metrics.items():
                all_metrics[metric_name].append(value)
                
        except Exception as e:
            print(f"⚠️ Error evaluating user {user_id}: {e}")
            continue
    
    avg_metrics = {}
    for metric_name, values in all_metrics.items():
        if values:
            avg_metrics[metric_name] = np.mean(values)
        else:
            avg_metrics[metric_name] = 0.0
    
    return avg_metrics

print("✅ Evaluation functions defined")

## 6. Đánh Giá Baseline Models

In [ ]:
# Prepare train user items for exclusion
train_user_items = train_df.groupby('user_id')['product_id'].apply(set).to_dict()

# Evaluate baseline models
k_values = [5, 10, 20]
max_eval_users = 500  # Limit for faster evaluation

print("🔬 Evaluating baseline models...")
print(f"📊 Evaluating on {min(len(test_ground_truth), max_eval_users)} users\n")

baseline_results = {}

# Popular items baseline
print("1️⃣ Evaluating Popular Items baseline...")
def popular_recommender(user_id: str, k: int, exclude: Set[str]) -> List[str]:
    return baseline.recommend_popular(user_id, k, exclude)

baseline_results['Popular'] = evaluate_recommender(
    popular_recommender,
    test_ground_truth,
    train_user_items,
    k_values,
    max_eval_users
)
print(f"   ✅ Popular Items - Precision@10: {baseline_results['Popular'].get('precision@10', 0):.4f}")

# User-based CF baseline
print("\n2️⃣ Evaluating User-Based CF baseline...")
def cf_recommender(user_id: str, k: int, exclude: Set[str]) -> List[str]:
    return baseline.recommend_user_based_cf(user_id, k, exclude)

baseline_results['User-Based CF'] = evaluate_recommender(
    cf_recommender,
    test_ground_truth,
    train_user_items,
    k_values,
    max_eval_users
)
print(f"   ✅ User-Based CF - Precision@10: {baseline_results['User-Based CF'].get('precision@10', 0):.4f}")

## 7. Visualization và So Sánh Kết Quả

In [ ]:
# Combine all results
all_results = baseline_results.copy()

# Create comparison DataFrame
results_df = pd.DataFrame(all_results).T

print("📊 Evaluation Results Summary:")
print("=" * 80)
display(results_df.round(4))

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Precision@K
precision_cols = [col for col in results_df.columns if col.startswith('precision@')]
if precision_cols:
    precision_df = results_df[precision_cols]
    precision_df.plot(kind='bar', ax=axes[0, 0], rot=45)
    axes[0, 0].set_title('Precision@K', fontsize=14, fontweight='bold')
    axes[0, 0].set_ylabel('Precision')
    axes[0, 0].legend(title='K')
    axes[0, 0].grid(axis='y', alpha=0.3)

# Recall@K
recall_cols = [col for col in results_df.columns if col.startswith('recall@')]
if recall_cols:
    recall_df = results_df[recall_cols]
    recall_df.plot(kind='bar', ax=axes[0, 1], rot=45, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    axes[0, 1].set_title('Recall@K', fontsize=14, fontweight='bold')
    axes[0, 1].set_ylabel('Recall')
    axes[0, 1].legend(title='K')
    axes[0, 1].grid(axis='y', alpha=0.3)

# NDCG@K
ndcg_cols = [col for col in results_df.columns if col.startswith('ndcg@')]
if ndcg_cols:
    ndcg_df = results_df[ndcg_cols]
    ndcg_df.plot(kind='bar', ax=axes[1, 0], rot=45, color=['#96CEB4', '#FFEAA7', '#DDA15E'])
    axes[1, 0].set_title('NDCG@K', fontsize=14, fontweight='bold')
    axes[1, 0].set_ylabel('NDCG')
    axes[1, 0].legend(title='K')
    axes[1, 0].grid(axis='y', alpha=0.3)

# MRR
if 'mrr' in results_df.columns:
    results_df['mrr'].plot(kind='bar', ax=axes[1, 1], color='#6C5CE7')
    axes[1, 1].set_title('Mean Reciprocal Rank (MRR)', fontsize=14, fontweight='bold')
    axes[1, 1].set_ylabel('MRR')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of all metrics
fig, ax = plt.subplots(figsize=(12, 6))

heatmap_metrics = ['precision@10', 'recall@10', 'ndcg@10', 'mrr']
available_metrics = [m for m in heatmap_metrics if m in results_df.columns]

if available_metrics:
    heatmap_data = results_df[available_metrics].T
    sns.heatmap(heatmap_data, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Score'})
    ax.set_title('Evaluation Metrics Heatmap', fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Models', fontsize=12)
    ax.set_ylabel('Metrics', fontsize=12)
    plt.tight_layout()
    plt.show()

## 8. Kết Luận và Gợi Ý

In [ ]:
print("\n" + "="*80)
print("📊 TÓM TẮT ĐÁNH GIÁ")
print("="*80)

if len(results_df) > 0:
    print("\n🏆 Best Models by Metric:")
    
    for metric in ['precision@10', 'recall@10', 'ndcg@10', 'mrr']:
        if metric in results_df.columns:
            best_model = results_df[metric].idxmax()
            best_score = results_df[metric].max()
            print(f"  {metric:20s}: {best_model:20s} ({best_score:.4f})")
    
    print("\n📈 Overall Ranking (Average Normalized Score):")
    
    normalized_df = results_df.copy()
    for col in normalized_df.columns:
        col_max = normalized_df[col].max()
        if col_max > 0:
            normalized_df[col] = normalized_df[col] / col_max
    
    normalized_df['overall_score'] = normalized_df.mean(axis=1)
    overall_ranking = normalized_df['overall_score'].sort_values(ascending=False)
    
    for rank, (model, score) in enumerate(overall_ranking.items(), 1):
        print(f"  {rank}. {model:20s}: {score:.4f}")

print("\n" + "="*80)
print("💡 GỢI Ý CẢI THIỆN:")
print("="*80)
print("1. Nếu Precision thấp: Tăng số lượng negative samples, cải thiện feature engineering")
print("2. Nếu Recall thấp: Tăng số lượng recommendations (K), cải thiện diversity")
print("3. Nếu NDCG thấp: Cải thiện ranking algorithm, thêm relevance signals")
print("4. Nếu Coverage thấp: Giảm popularity bias, thêm exploration mechanisms")
print("5. Xem xét hybrid approaches để kết hợp strengths của nhiều models")
print("="*80)

In [ ]:
# Export results to CSV
output_file = "evaluation_results.csv"
results_df.to_csv(output_file)
print(f"✅ Results exported to {output_file}")

# Display final results table
display(results_df.round(4))